In [ ]:
!apt-get install -y php-cli > /dev/null 2>&1
!php -v
print("✅ PHP ready")

PHP 8.1.2-1ubuntu2.23 (cli) (built: Jan  7 2026 08:37:41) (NTS)
Copyright (c) The PHP Group
Zend Engine v4.1.2, Copyright (c) Zend Technologies
    with Zend OPcache v8.1.2-1ubuntu2.23, Copyright (c), by Zend Technologies
✅ PHP ready


In [ ]:
from google.colab import files
uploaded = files.upload()
print("✅ Uploaded:", list(uploaded.keys()))

Saving index.php to index.php
✅ Uploaded: ['index.php']


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version
print("✅ Tunnel tool ready")

cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)
✅ Tunnel tool ready


In [ ]:
import subprocess, time, threading

# Start PHP server
php = subprocess.Popen(
    ["php", "-S", "0.0.0.0:8080", "index.php"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)
print("✅ PHP server started")

# Start cloudflare tunnel and capture URL
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("⏳ Getting your link (takes up to 20 seconds)...")

# Read stderr line by line until we find the URL
url_found = False
for i in range(60):
    line = tunnel.stderr.readline()
    if "trycloudflare.com" in line:
        # Extract just the URL
        import re
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            print("\n" + "="*60)
            print("🚀  InsightGuard is LIVE! Open this link:")
            print(f"\n   👉  {url}\n")
            print("="*60)
            print("\nLeave this cell running while you use the tool.")
            url_found = True
            break

if not url_found:
    print("⚠️  URL not found automatically. Check above for a https://...trycloudflare.com link")

✅ PHP server started
⏳ Getting your link (takes up to 20 seconds)...

🚀  InsightGuard is LIVE! Open this link:

   👉  https://legs-village-netscape-das.trycloudflare.com


Leave this cell running while you use the tool.


In [ ]:
import subprocess
result = subprocess.run(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8080"],
    capture_output=True, text=True, timeout=25
)
print("STDOUT:", result.stdout[:500])
print("STDERR:", result.stderr[:500])

TimeoutExpired: Command '['./cloudflared', 'tunnel', '--url', 'http://localhost:8080']' timed out after 25 seconds

In [ ]:
# This runs PHP locally and shows the output directly in Colab
import subprocess

result = subprocess.run(
    ["php", "-f", "index.php"],
    capture_output=True, text=True,
    env={"REQUEST_METHOD": "GET", "PATH": "/usr/bin:/bin"}
)

from IPython.display import HTML, display
display(HTML(result.stdout))